# သင်ခန်းစာ 03 - Agentic ဒီဇိုင်းပုံစံများ

ဒီသင်ခန်းစာမှာ ကျွန်တော်တို့ အကျိုးရှိစေတဲ့ AI အေးဂျင့်တွေ တည်ဆောက်ဖို့ အခြေခံဒီဇိုင်းပုံစံသုံးခုကို လေ့လာမှာဖြစ်ပါတယ်။

1. **ရှင်းလင်းတဲ့ အေးဂျင့်ညွှန်ကြားချက်များ** — အေးဂျင့်လုပ်ဆောင်ချက်ကို လမ်းညွှန်ပေးဖို့ တိကျပြီး တာဝန်သက်သက်သာသာ ပညာပေးစာလုံးချုပ်များကို ဖန်တီးခြင်း
2. **Pydantic မော်ဒယ်များဖြင့် အစီအစဉ်တကျ ထုတ်လုပ်မှု** — အေးဂျင့်များက မျှော်မှန်းထားတဲ့၊ အတည်ပြုထားတဲ့ ဒေတာကို ပြန်လည်ပေးပို့စေခြင်း
3. **တစ်ခုတည်းတာဝန်ရှိတဲ့ အေးဂျင့်များ** — တစ်ခုချင်းစီက အလုပ်တစ်ခုကို ကောင်းစွာ လုပ်ဆောင်နိုင်ဖို့ အာရုံစိုက်သေချာ ဒီဇိုင်းရေးဆွဲခြင်း

ဒီပုံစံတိုင်းကို **ခရီးသွားဒေသ အကြံပြုစနစ်** ပြဿနာအတွက် သုံးပြီး နေရာအကြံပြုခြင်း၊ ရရှိနိုင်မှု စစ်ဆေးခြင်းနဲ့ လက်တွေ့စီမံခန့်ခွဲမှုကို တစ်ဆင့်ချင်းတိုးတက်စေမယ့် စနစ်တစ်ခု တည်ဆောက်မှာဖြစ်ပါတယ်။


## စတင်ဆောင်ရွက်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ပုံစံ ၁: ကိုယ်စားလှယ်အားရှင်းလင်းသေချာသောညွှန်ကြားချက်များ

အကျိုးသက်ရောက်မှုအမြင့်ဆုံးသောပုံစံမှာလည်း အလွန်ရိုးရှင်းသည်- ကိုယ်စားလှယ်အတွက် ရှင်းလင်းပြီးအသေးစိတ်ညွှန်ကြားချက်များရေးသားခြင်းဖြစ်သည်။

ကောင်းမွန်သည့်ညွှန်ကြားချက်များသည် အောက်ပါအချက်များကို သတ်မှတ်သည်-
- **ဘယ်သူ** ကိုယ်စားလှယ်ပါသည် (ပုဂ္ဂိုလ်ရေးနှင့် အသံ)
- **ဘာ** ကို ပြုလုပ်သင့်သည် (ခြေလှမ်းလိုက်တာဝန်များ)
- **ဘယ်လို** ကိုယ်စားလှယ် ဖြစ်ရမည် (ကန့်သတ်ချက်များနှင့် စတိုင်)

အောက်တွင်၊ ကျွန်ုပ်တို့သည် တစ်ဦးချင်းပြန်ကြားချက်တစ်ခုချင်းစီကို ပုံဖော်သတ်မှတ်သည့် သွားလာမှုဆိုင်ရာစာရေးသူကိုယ်စားလှယ်တစ်ဦးကို ဖန်တီးထားသည်။


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## ပုံစံ ၂: Pydantic မော်ဒယ်များဖြင့် ဖွဲ့စည်းထားသော ထွက်ရှိမှု

အလွတ်သက်သက်စာသားသည် စကားပြောဆိုရာတွင် အသုံးဝင်ပေမယ့် အောက်လွှာစနစ်များအတွက် ဖွဲ့စည်းထားသော ဒေတာများလိုအပ်သည်။
**Pydantic မော်ဒယ်များ**ကို **ကိရိယာလုပ်ဆောင်ချက်** တစ်ခုနှင့် တွဲဖက်၍၊

- ကိုယ်စားလှယ်၏ ထွက်ရှိမှုအတွက် တိကျသော စခရမ်းကို သတ်မှတ်နိုင်သည်
- တုံ့ပြန်ချက်များကို အလိုအလျောက် စစ်ဆေးနိုင်သည်
- ကိုယ်စားလှယ်ရလဒ်များကို လုပ်ထုံးလုပ်နည်းအတွင်း ယုံကြည်စိတ်ချစွာ ဖြည့်စွက်နိုင်သည်

အကောင်အထည်ဖော်ရန် အဓိကမှာ ကိုယ်စားလှယ်ကို စစ်ဆေးစေရန် `response_format` ကို ပေးပို့ခြင်းဖြစ်သည်။ ၎င်းသည်
မော်ဒယ်ကို အလွတ်သက်သက်စာသားအစား စစ်ဆေးပြီးနောက် `TravelRecommendations` အရာဝတ္ထု (`response.value` တွင်ရနိုင်သည်) ပြန်လည်ပေးစေနိုင်သည်။
`get_destination_details` ကိရိယာလည်း ပုံစံသတ်မှတ်ထားသော `DestinationRecommendation` ကိုပြန်လည်ပေးသောကြောင့်
ဒေတာများသည် အစမှအဆုံး ဖွဲ့စည်းထားသည်။


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## ပုံစံ ၃။ တစ်ကြိမ်တည်းတာဝန်ရှိသူများ

တင်းကျပ်သောတာဝန်များကို အာရုံစူးစိုက်မှုရှိသည့် အေးဂျင့်များစွာဖြင့် ခွဲခြားတာဝန်ပေးခြင်းအားဖြင့် အကျိုးရှိစေသည်။

- **နေရာနှင့်ရရှိနိုင်မှုများအကြောင်း သိရှိသူ** တစ်ဦး
- သေချာစီမံခန့်ခွဲသူ လေယာဉ်၊ ဟိုတယ်နှင့် ခရီးစဉ်များကို ကူညီစီမံခြင်း

၎င်းသည် *စိတ်ပူပန်မှုများခွဲခြားစေခြင်း* ဆိုသော ဆော့ဖ်ဝဲ အင်ဂျင်နီယာနည်းဗျူဟာနှင့် တူသည် — အေးဂျင့်တိုင်းကို စမ်းသပ်ရန်၊ ထိန်းသိမ်းရန်နှင့် တိုးတက်အောင်ပြုပြင်ရန် ပိုမိုလွယ်ကူသည်။


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## အကျဥ်းချုပ်

ဒီသင်ခန်းစာမှာ ကျွန်ုပ်တို့သည် ခရီးသွားအကြံပြုရေးစနစ်ဇာတ်ကောင်တစ်ခုအတွက် အေးဂျင့် ဒီဇိုင်းပုံစံသုံးခုကို အသုံးပြုခဲ့သည်။

| ပုံစံ | အဓိကစိတ်ကူး | အကျိုးကျေးဇူး |
|---|---|---|
| **ရှင်းရှင်းလင်းလင်းညွှန်ကြားချက်များ** | ပုဂ္ဂိုလ်ရေး၊ တာဝန်များနှင့် ကန့်သတ်ချက်များကို အစပိုင်းမှာ သတ်မှတ်ခြင်း | တည်ငြိမ်ပြီး အမှတ်တံဆိပ်နှင့် ကိုက်ညီသော အေးဂျင့်အပြုအမူ |
| **ဖွဲ့စည်းထားသော ထုတ်လွှင့်ချက်** | Pydantic မော်ဒယ်များကို တုံ့ပြန်ချက်ပုံစံအဖြစ် အသုံးပြုခြင်း | အသိအမှတ်ပြု၍ ကွန်ပျူတာဖတ်နိုင်သော ရလဒ်များ |
| **တစ်ခုတည်းသော တာဝန်အလေးပြုခြင်း** | အေးဂျင့် တစ်ဦးချင်းစီအား တာဝန်တစ်ခုအာရုံစိုက်ပေးခြင်း | စမ်းသပ်ရလွယ်ကူခြင်း၊ ထိန်းသိမ်းရလွယ်ခြင်းနှင့် ပေါင်းစပ်ရလွယ်ခြင်း |

ဤပုံစံများသည် သဘာဝအတိုင်း ပေါင်းစပ်နိုင်ပြီး - တစ်ခုတည်းတာဝန်ရှိသော အေးဂျင့်တစ်ဦးထဲတွင် ရှင်းရှင်းလင်းလင်းညွှန်ကြားချက်များနှင့် ဖွဲ့စည်းထားသော ထုတ်လွှင့်ချက်များကို ပေါင်းစပ်နိုင်ပြီး ခိုင်မာပြီး ထုတ်လုပ်မှုအဆင်သင့် စနစ်များကို တည်ဆောက်နိုင်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
